# Nat Gas ML Extension

Walk-forward bake-off: OLS, ARMA-GARCH, Random Forest, LSTM. 71 monthly observations, 35 OOS predictions.


In [1]:
import sys
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from itertools import combinations
from scipy.stats import t as student_t

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from src.models import SignificantOLSModel, GARCHModel

DATA = ROOT / 'data' / 'Book1.1.xlsx'
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
TARGET = 'NG_Return'
TRAIN_WINDOW = 36


## Load data

In [2]:
raw = pd.read_excel(DATA, index_col=0, parse_dates=True).dropna().sort_index()
print(f'shape={raw.shape}, range={raw.index.min().date()} to {raw.index.max().date()}')


shape=(71, 25), range=2020-01-31 to 2025-11-30


## OLS walk-forward (uses raw column names)

In [3]:
rows = []
for i in range(TRAIN_WINDOW, len(raw)):
    train, test = raw.iloc[:i], raw.iloc[i:i+1]
    m = SignificantOLSModel().fit(train)
    rows.append({'date': test.index[0], 'actual': test[TARGET].values[0], 'ols_pred': m.predict(test)[0]})
ols = pd.DataFrame(rows).set_index('date')
ols.to_parquet(RESULTS / 'ols_predictions.parquet')
hit = (np.sign(ols.actual) == np.sign(ols.ols_pred)).mean()
print(f'OLS: n={len(ols)}, hit={hit:.3f}')


OLS: n=35, hit=0.629


## GARCH walk-forward

In [4]:
rows = []
for i in range(TRAIN_WINDOW, len(raw)):
    train_r, test = raw.iloc[:i][TARGET], raw.iloc[i:i+1]
    try:
        m = GARCHModel('AR', 1, 1, 1).fit(train_r, dist='normal')
        mean_fc, _ = m.forecast(horizon=1)
        pred = float(np.asarray(mean_fc).flatten()[0]) / 100.0
        rows.append({'date': test.index[0], 'actual': test[TARGET].values[0], 'garch_pred': pred})
    except Exception:
        pass
garch = pd.DataFrame(rows).set_index('date')
garch.to_parquet(RESULTS / 'garch_predictions.parquet')
hit = (np.sign(garch.actual) == np.sign(garch.garch_pred)).mean()
print(f'GARCH: n={len(garch)}, hit={hit:.3f}')


GARCH: n=35, hit=0.400


## Random Forest (with feature engineering)

In [5]:
# Stripped column names for RF/LSTM
df = raw.copy()
df.columns = [c.strip() for c in df.columns]

# Engineered features
for k in [1, 3, 6, 12]:
    df[f'ret_lag_{k}'] = df[TARGET].shift(k)
df['vol_3m'] = df[TARGET].rolling(3).std().shift(1)
df['vol_12m'] = df[TARGET].rolling(12).std().shift(1)
df['mom_12m'] = df[TARGET].rolling(12).sum().shift(1)

FEATS = ['Coal Price Index', 'HenryHub Spot', 'Storage vs 5yr Avg', 'EIA Storage Change',
         'Net Trade Balance', 'Carbon EUA Futures Price',
         'ret_lag_1', 'ret_lag_3', 'ret_lag_6', 'ret_lag_12',
         'vol_3m', 'vol_12m', 'mom_12m']
df_rf = df[FEATS + [TARGET]].dropna()

rows = []
tscv = TimeSeriesSplit(n_splits=5)
last_imp = None
for tr, te in tscv.split(df_rf):
    X_tr, y_tr = df_rf[FEATS].iloc[tr], df_rf[TARGET].iloc[tr]
    X_te, y_te = df_rf[FEATS].iloc[te], df_rf[TARGET].iloc[te]
    rf = RandomForestRegressor(n_estimators=500, max_depth=8, min_samples_leaf=20,
                               max_features='sqrt', random_state=42, n_jobs=-1).fit(X_tr, y_tr)
    pred = rf.predict(X_te)
    for d, a, p in zip(X_te.index, y_te.values, pred):
        rows.append({'date': d, 'actual': a, 'rf_pred': p})
    last_imp = pd.DataFrame({
        'gini': rf.feature_importances_,
        'permutation': permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=42).importances_mean,
    }, index=FEATS)

rf_df = pd.DataFrame(rows).set_index('date')
rf_df.to_parquet(RESULTS / 'rf_predictions.parquet')
last_imp.sort_values('gini', ascending=False).to_csv(RESULTS / 'rf_feature_importance.csv')
print(f'RF: n={len(rf_df)}, hit={(np.sign(rf_df.actual) == np.sign(rf_df.rf_pred)).mean():.3f}')
print(last_imp.sort_values('gini', ascending=False).head(5))


RF: n=45, hit=0.444
                    gini  permutation
Coal Price Index     0.0          0.0
HenryHub Spot        0.0          0.0
Storage vs 5yr Avg   0.0          0.0
EIA Storage Change   0.0          0.0
Net Trade Balance    0.0          0.0


## LSTM (PyTorch, 12-month window)

In [6]:
WINDOW = 12
HIDDEN = 64
EPOCHS = 50
PATIENCE = 5
torch.manual_seed(42)
np.random.seed(42)

LSTM_FEATS = ['Coal Price Index', 'HenryHub Spot', 'Storage vs 5yr Avg', 'VIX',
              'Net Trade Balance', 'Carbon EUA Futures Price', TARGET]
df_l = df[LSTM_FEATS].dropna().copy()

class LSTMReg(nn.Module):
    def __init__(self, n_feat):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, HIDDEN, num_layers=2, dropout=0.3, batch_first=True)
        self.head = nn.Linear(HIDDEN, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)

def make_windows(arr, window):
    X = np.stack([arr[i-window:i] for i in range(window, len(arr))])
    return X

rows = []
tscv = TimeSeriesSplit(n_splits=5)
for tr, te in tscv.split(df_l):
    train_df = df_l.iloc[tr]
    test_df = df_l.iloc[te]
    if len(train_df) < WINDOW + 5 or len(test_df) < 1:
        continue
    mu = train_df.mean()
    sd = train_df.std().replace(0, 1)
    train_z = (train_df - mu) / sd
    test_z = (test_df - mu) / sd
    X_tr = make_windows(train_z.values, WINDOW)
    y_tr = train_df[TARGET].values[WINDOW:]
    full_z = pd.concat([train_z.iloc[-WINDOW:], test_z]).values
    X_te = make_windows(full_z, WINDOW)
    y_te = test_df[TARGET].values

    model = LSTMReg(len(LSTM_FEATS))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    best, best_state, bad = float('inf'), None, 0
    val_split = max(1, len(X_tr) // 5)
    Xt, yt = torch.tensor(X_tr[:-val_split], dtype=torch.float32), torch.tensor(y_tr[:-val_split], dtype=torch.float32)
    Xv, yv = torch.tensor(X_tr[-val_split:], dtype=torch.float32), torch.tensor(y_tr[-val_split:], dtype=torch.float32)

    for ep in range(EPOCHS):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(Xt), yt)
        loss.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            v = loss_fn(model(Xv), yv).item()
        if v < best:
            best, best_state, bad = v, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    if best_state:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X_te, dtype=torch.float32)).numpy()
    for d, a, p in zip(test_df.index, y_te, pred):
        rows.append({'date': d, 'actual': a, 'lstm_reg': p})

lstm = pd.DataFrame(rows).set_index('date')
lstm.to_parquet(RESULTS / 'lstm_predictions.parquet')
print(f'LSTM: n={len(lstm)}, hit={(np.sign(lstm.actual) == np.sign(lstm.lstm_reg)).mean():.3f}')


LSTM: n=44, hit=0.500


## 4-model comparison + Diebold-Mariano + ensembles

In [7]:
def metrics(actual, pred, cost=0.001):
    err = actual - pred
    rmse = float(np.sqrt((err**2).mean()))
    mae = float(err.abs().mean())
    hit = float((np.sign(actual) == np.sign(pred)).mean())
    sig = np.sign(pred)
    strat = sig * actual
    trades = sig.diff().abs().fillna(0.0)
    strat_net = strat - trades * cost
    sharpe = float(strat_net.mean() / strat_net.std() * np.sqrt(12)) if strat_net.std() > 0 else 0.0
    eq = (1.0 + strat_net).cumprod()
    dd = float((eq / eq.cummax() - 1.0).min())
    return {'RMSE': rmse, 'MAE': mae, 'Hit': hit, 'Sharpe': sharpe, 'MaxDD': dd}

def dm_test(actual, p1, p2):
    e1, e2 = (actual - p1).values, (actual - p2).values
    d = e1**2 - e2**2
    n = len(d)
    if n < 4 or d.var() <= 0:
        return float('nan'), float('nan')
    dm = d.mean() / np.sqrt(d.var() / n) * np.sqrt((n + 1 - 2) / n)
    p = 2.0 * (1.0 - student_t.cdf(abs(dm), df=n-1))
    return float(dm), float(p)

# Stitch on common dates
all_df = ols[['actual', 'ols_pred']].join(garch[['garch_pred']], how='inner') \
    .join(rf_df[['rf_pred']], how='inner').join(lstm[['lstm_reg']], how='inner').dropna()
preds = ['ols_pred', 'garch_pred', 'rf_pred', 'lstm_reg']

rows = [{'Model': c, **metrics(all_df.actual, all_df[c])} for c in preds]
all_df['ensemble_eq'] = all_df[preds].mean(axis=1)
rows.append({'Model': 'ensemble_eq', **metrics(all_df.actual, all_df.ensemble_eq)})

# Stacked logistic
X = all_df[preds].values
y = (all_df.actual > 0).astype(int).values
stack = pd.Series(index=all_df.index, dtype=float)
for tr, te in TimeSeriesSplit(n_splits=5).split(X):
    meta = LogisticRegression(max_iter=1000).fit(X[tr], y[tr])
    stack.iloc[te] = meta.predict_proba(X[te])[:, 1] - 0.5
all_df['ensemble_stack'] = stack
sd = all_df.dropna(subset=['ensemble_stack'])
rows.append({'Model': 'ensemble_stack', **metrics(sd.actual, sd.ensemble_stack)})

summary = pd.DataFrame(rows).set_index('Model').round(4)
summary.to_csv(RESULTS / 'model_comparison.csv')
print(summary)

dm_rows = [{'A': a, 'B': b, **dict(zip(['DM', 'p'], dm_test(all_df.actual, all_df[a], all_df[b])))}
           for a, b in combinations(preds, 2)]
dm_df = pd.DataFrame(dm_rows).round(4)
dm_df.to_csv(RESULTS / 'dm_tests.csv', index=False)
print('\nDiebold-Mariano:')
print(dm_df.to_string(index=False))


                  RMSE     MAE     Hit  Sharpe   MaxDD
Model                                                 
ols_pred        0.2431  0.1879  0.6286  0.9376 -0.5909
garch_pred      0.2601  0.1953  0.4000 -1.4942 -0.9833
rf_pred         0.2624  0.2014  0.4286  0.0997 -0.8171
lstm_reg        0.2561  0.2016  0.4857  0.4162 -0.7619
ensemble_eq     0.2390  0.1816  0.6571  1.2115 -0.5909
ensemble_stack  0.2862  0.2338  0.5200  0.0466 -0.6366

Diebold-Mariano:
         A          B      DM      p
  ols_pred garch_pred -0.4467 0.6580
  ols_pred    rf_pred -0.4911 0.6265
  ols_pred   lstm_reg -0.3800 0.7063
garch_pred    rf_pred -0.4325 0.6681
garch_pred   lstm_reg  0.5539 0.5833
   rf_pred   lstm_reg  0.7075 0.4841
